#AI-Based Movie Personalization Engine - Visualization Engine

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

from IPython.display import display, clear_output

import warnings

warnings.filterwarnings("ignore")

sns.set_theme(
    style="whitegrid",
    palette="deep",
    rc={
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.figsize": (12, 6),
        "axes.titlesize": 14,
        "axes.labelsize": 12
    }
)


# ============================================================
# LOAD DATA
# ============================================================

movies = pd.read_csv("movies.csv")
processed_movies = pd.read_csv("processed_movies.csv")
viewing_history = pd.read_csv("clean_viewing_history.csv")
user_stats = pd.read_csv("user_stats.csv")

analysis_genres = pd.read_csv("analysis_genres.csv")
analysis_genre_ratings = pd.read_csv("analysis_genre_ratings.csv")
analysis_genre_views = pd.read_csv("analysis_genre_views.csv")
analysis_age_groups = pd.read_csv("analysis_age_groups.csv")
analysis_movie_performance = pd.read_csv(
    "analysis_movie_performance.csv"
)
analysis_user_stats = pd.read_csv(
    "analysis_user_stats.csv"
)


# ============================================================
# PREPARE DATA
# ============================================================

genre_distribution = (
    movies["genre"]
    .value_counts()
    .reset_index()
)

genre_distribution.columns = [
    "genre",
    "movie_count"
]


genre_ratings = analysis_genre_ratings.copy()
genre_views = analysis_genre_views.copy()
age_groups = analysis_age_groups.copy()
movie_performance = analysis_movie_performance.copy()
user_statistics = analysis_user_stats.copy()


# ============================================================
# PREPARE VIEWING HISTORY
# ============================================================

timestamp_candidates = [
    "timestamp",
    "datetime",
    "date",
    "watched_at",
    "viewed_at",
    "viewing_time"
]

timestamp_column = None

for column in timestamp_candidates:

    if column in viewing_history.columns:
        timestamp_column = column
        break


if timestamp_column is not None:

    viewing_history[timestamp_column] = pd.to_datetime(
        viewing_history[timestamp_column],
        errors="coerce"
    )

    viewing_history["hour"] = (
        viewing_history[timestamp_column].dt.hour
    )


if "genre" not in viewing_history.columns:

    if "movie_id" in viewing_history.columns:

        movie_genres = movies[
            ["movie_id", "genre"]
        ].drop_duplicates()

        viewing_history = viewing_history.merge(
            movie_genres,
            on="movie_id",
            how="left"
        )


# ============================================================
# DASHBOARD
# ============================================================

class AnalyticsDashboard:

    def __init__(
        self,
        genre_distribution,
        genre_ratings,
        genre_views,
        viewing_history,
        age_groups,
        movie_performance,
        user_statistics
    ):

        self.genre_distribution = genre_distribution
        self.genre_ratings = genre_ratings
        self.genre_views = genre_views
        self.viewing_history = viewing_history
        self.age_groups = age_groups
        self.movie_performance = movie_performance
        self.user_statistics = user_statistics

        self.out = widgets.Output()


        self.metric_selector = widgets.Dropdown(

            options=[
                "User Interest Distribution",
                "Genre Ratings",
                "Genre Views",
                "Activity Heatmap",
                "Age Group Analysis",
                "Movie Performance",
                "User Statistics"
            ],

            value="User Interest Distribution",

            description="View:",

            layout={
                "width": "350px"
            }
        )


        self.genre_filter = widgets.SelectMultiple(

            options=sorted(
                self.genre_distribution[
                    "genre"
                ].dropna().unique().tolist()
            ),

            value=tuple(
                self.genre_distribution[
                    "genre"
                ].dropna().unique().tolist()
            ),

            description="Genres:",

            layout={
                "width": "300px"
            }
        )


        self.metric_selector.observe(
            self.update_dashboard,
            names="value"
        )

        self.genre_filter.observe(
            self.update_dashboard,
            names="value"
        )


    # ========================================================
    # USER INTEREST DISTRIBUTION
    # ========================================================

    def render_interest_distribution(self):

        filtered = self.genre_distribution[
            self.genre_distribution["genre"].isin(
                self.genre_filter.value
            )
        ]

        if filtered.empty:
            print("No genre data available.")
            return


        fig, ax = plt.subplots(
            figsize=(12, 6)
        )


        sns.barplot(
            data=filtered,
            x="genre",
            y="movie_count",
            ax=ax
        )


        ax.set_title(
            "Movie Distribution by Genre",
            fontweight="bold"
        )

        ax.set_xlabel("Genre")
        ax.set_ylabel("Number of Movies")

        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()


    # ========================================================
    # GENRE RATINGS
    # ========================================================

    def render_genre_ratings(self):

        if self.genre_ratings.empty:
            print("No genre rating data available.")
            return


        genre_column = self.genre_ratings.columns[0]
        value_column = self.genre_ratings.columns[1]


        filtered = self.genre_ratings[
            self.genre_ratings[genre_column].isin(
                self.genre_filter.value
            )
        ]


        if filtered.empty:
            print("No matching genre data.")
            return


        fig, ax = plt.subplots(
            figsize=(12, 6)
        )


        sns.barplot(
            data=filtered,
            x=genre_column,
            y=value_column,
            ax=ax
        )


        ax.set_title(
            "Average Rating by Genre",
            fontweight="bold"
        )

        ax.set_xlabel("Genre")
        ax.set_ylabel("Average Rating")

        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()


    # ========================================================
    # GENRE VIEWS
    # ========================================================

    def render_genre_views(self):

        if self.genre_views.empty:
            print("No genre viewing data available.")
            return


        genre_column = self.genre_views.columns[0]
        value_column = self.genre_views.columns[1]


        filtered = self.genre_views[
            self.genre_views[genre_column].isin(
                self.genre_filter.value
            )
        ]


        if filtered.empty:
            print("No matching genre data.")
            return


        fig, ax = plt.subplots(
            figsize=(12, 6)
        )


        sns.barplot(
            data=filtered,
            x=genre_column,
            y=value_column,
            ax=ax
        )


        ax.set_title(
            "Viewing Activity by Genre",
            fontweight="bold"
        )

        ax.set_xlabel("Genre")
        ax.set_ylabel("Number of Views")

        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()


    # ========================================================
    # ACTIVITY HEATMAP
    # ========================================================

    def render_activity_heatmap(self):

        if "hour" not in self.viewing_history.columns:

            print(
                "No timestamp column was found "
                "in the viewing history."
            )

            print(
                "Available columns:"
            )

            print(
                self.viewing_history.columns.tolist()
            )

            return


        if "genre" not in self.viewing_history.columns:

            print(
                "Genre information is not available."
            )

            return


        activity = (
            self.viewing_history
            .dropna(
                subset=["hour", "genre"]
            )
            .groupby(
                ["hour", "genre"]
            )
            .size()
            .reset_index(
                name="activity"
            )
        )


        if activity.empty:

            print(
                "No activity data available."
            )

            return


        heatmap_data = activity.pivot(
            index="hour",
            columns="genre",
            values="activity"
        ).fillna(0)


        selected_genres = [
            genre
            for genre in self.genre_filter.value
            if genre in heatmap_data.columns
        ]


        heatmap_data = heatmap_data[
            selected_genres
        ]


        if heatmap_data.empty:

            print(
                "No matching genres found."
            )

            return


        fig, ax = plt.subplots(
            figsize=(12, 7)
        )


        sns.heatmap(
            heatmap_data,
            annot=True,
            fmt=".0f",
            cmap="YlOrRd",
            ax=ax
        )


        ax.set_title(
            "Viewing Activity by Hour and Genre",
            fontweight="bold"
        )

        ax.set_xlabel("Genre")
        ax.set_ylabel("Hour of Day")

        plt.tight_layout()
        plt.show()


    # ========================================================
    # AGE GROUP ANALYSIS
    # ========================================================

    def render_age_groups(self):

        if self.age_groups.empty:

            print(
                "No age-group data available."
            )

            return


        x_column = self.age_groups.columns[0]
        y_column = self.age_groups.columns[1]


        fig, ax = plt.subplots(
            figsize=(12, 6)
        )


        sns.barplot(
            data=self.age_groups,
            x=x_column,
            y=y_column,
            ax=ax
        )


        ax.set_title(
            "User Distribution by Age Group",
            fontweight="bold"
        )

        ax.set_xlabel("Age Group")
        ax.set_ylabel("Number of Users")

        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()


    # ========================================================
    # MOVIE PERFORMANCE
    # ========================================================

    def render_movie_performance(self):

        if self.movie_performance.empty:

            print(
                "No movie performance data available."
            )

            return


        numeric_columns = (
            self.movie_performance
            .select_dtypes(
                include=np.number
            )
            .columns
        )


        if len(numeric_columns) == 0:

            print(
                "No numeric performance data available."
            )

            return


        value_column = numeric_columns[-1]
        movie_column = self.movie_performance.columns[0]


        top_movies = (
            self.movie_performance
            .nlargest(
                10,
                value_column
            )
        )


        fig, ax = plt.subplots(
            figsize=(12, 6)
        )


        sns.barplot(
            data=top_movies,
            x=movie_column,
            y=value_column,
            ax=ax
        )


        ax.set_title(
            "Top Performing Movies",
            fontweight="bold"
        )

        ax.set_xlabel(movie_column)
        ax.set_ylabel(value_column)

        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()


    # ========================================================
    # USER STATISTICS
    # ========================================================

    def render_user_statistics(self):

        if self.user_statistics.empty:

            print(
                "No user statistics available."
            )

            return


        numeric_columns = (
            self.user_statistics
            .select_dtypes(
                include=np.number
            )
            .columns
        )


        if len(numeric_columns) == 0:

            print(
                "No numeric user statistics available."
            )

            return


        value_column = numeric_columns[-1]


        fig, ax = plt.subplots(
            figsize=(12, 6)
        )


        sns.histplot(
            data=self.user_statistics,
            x=value_column,
            kde=True,
            ax=ax
        )


        ax.set_title(
            "Distribution of User Activity",
            fontweight="bold"
        )

        ax.set_xlabel(value_column)
        ax.set_ylabel("Number of Users")

        plt.tight_layout()
        plt.show()


    # ========================================================
    # DASHBOARD CONTROLLER
    # ========================================================

    def update_dashboard(self, *args):

        with self.out:

            clear_output(
                wait=True
            )


            selected_view = (
                self.metric_selector.value
            )


            if selected_view == (
                "User Interest Distribution"
            ):

                self.render_interest_distribution()


            elif selected_view == "Genre Ratings":

                self.render_genre_ratings()


            elif selected_view == "Genre Views":

                self.render_genre_views()


            elif selected_view == "Activity Heatmap":

                self.render_activity_heatmap()


            elif selected_view == "Age Group Analysis":

                self.render_age_groups()


            elif selected_view == "Movie Performance":

                self.render_movie_performance()


            elif selected_view == "User Statistics":

                self.render_user_statistics()


    # ========================================================
    # DISPLAY DASHBOARD
    # ========================================================

    def display(self):

        self.update_dashboard()


        controls = widgets.HBox([
            self.metric_selector,
            self.genre_filter
        ])


        ui = widgets.VBox([

            widgets.HTML(
                "<h2>Movie Maestro Analytics Dashboard</h2>"
            ),

            controls,

            self.out

        ])


        display(ui)


# ============================================================
# LAUNCH DASHBOARD
# ============================================================

dashboard = AnalyticsDashboard(

    genre_distribution=genre_distribution,

    genre_ratings=genre_ratings,

    genre_views=genre_views,

    viewing_history=viewing_history,

    age_groups=age_groups,

    movie_performance=movie_performance,

    user_statistics=user_statistics
)


dashboard.display()
